# BIST harness: Gemma 3 4B QLoRA eğitimi (Colab T4)
Adımlar: (1) GPU kontrolü, (2) Unsloth kurulumu, (3) `colab_bundle.zip` yükle, (4) kuru çalıştırma (tokenizasyon/maske istatistiği), (5) eğitim, (6) sonuçları indir.
Ayrıntı ve sorun giderme: `training/README.md`. Zip, `python -m training.colab.make_bundle` ile hazırlanır (eğitim kodu + train.jsonl + val.jsonl).

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available(), "| bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None)

In [ ]:
!pip install -q unsloth

In [ ]:
from google.colab import files
up = files.upload()   # colab_bundle.zip
!unzip -o -q colab_bundle.zip
!ls -la . training/colab

## Kuru çalıştırma (model yüklemeden)
Kaç örnek bağlam sınırına (8192) takılıyor, eğitilen token payı ne kadar? `data_stats.json` yazılır.

In [ ]:
!python training/colab/train_qlora.py --train train.jsonl --val val.jsonl --out outputs/v1 --dry-run

## Eğitim
T4 üzerinde ~1-2 saat (2 epoch, ~500 örnek, ortalama ~4,4 bin token). Kayıp yalnızca asistan turlarında hesaplanır. `--gguf q4_k_m` eğitim sonunda Ollama için GGUF üretir.

In [ ]:
!python training/colab/train_qlora.py --train train.jsonl --val val.jsonl --out outputs/v1 --epochs 2 --gguf q4_k_m

In [ ]:
!ls -la outputs/v1 outputs/v1/gguf
!cat outputs/v1/data_stats.json | head -40
!cat outputs/v1/train_result.json

## Sonuçları indir
`result.zip`: adaptör + GGUF + istatistikler. GGUF'u Bilgisayar 1'de `training/ollama/gemma3-bist-q4_k_m.gguf` olarak kaydedin (training/README.md, bölüm 5).

In [ ]:
!cd outputs/v1 && zip -r -q ../../result.zip adapter gguf data_stats.json train_result.json
from google.colab import files
files.download("result.zip")